# Giai đoạn 3 – Bước 8C: LangGraph Prediction Agent (do bạn tự chạy)

**Mục tiêu:** Kết nối Orchestrator tối giản → kiểm tra đầu vào → Prediction Agent → FastAPI nghiên cứu 8B → kiểm tra response → trả kết quả an toàn.

- Chỉ sử dụng dữ liệu **giả lập**; không nhập dữ liệu sức khỏe cá nhân.
- Không train, không `.fit()`, không nạp `joblib`, không đọc lại Test.
- Không khởi động API thay bạn; **bạn** phải mở terminal chạy 8B trước khi làm kiểm tra live.
- Bước này mới tích hợp **một** Agent theo workflow LangGraph. Data Analysis/Knowledge/Consultation và hệ thống 5 Agent đầy đủ thuộc các giai đoạn tiếp theo.

## 1. Cài thư viện trong đúng kernel (do bạn quyết định chạy ô này)

Chạy một lần nếu chưa cài `langgraph`, `requests`, `pydantic`. Không thay đổi môi trường chứa model nếu không cần. Kiểm tra yêu cầu phiên bản của Python và scikit-learn trong server 8B.

In [1]:
%pip install langgraph requests pydantic

  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached zstandard-0.25.0-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
  Using cached jsonpointer-3.7-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/577.2 kB ? eta -:--:--
   ---------------------------------------- 577.2/577.2 kB 2.9 MB/s  0:00:00
   ---------------------------------------- 0.0/940.8 kB ? eta -:--:--
   ---------------------------------------- 940.8/940.8 kB 6.0 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached certifi-2026.10.23-py3-none-any.whl (135 kB)
Using cached jsonpointer-3.7-py3-none-any.whl (9.4 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Using cached zstandard-0.25.0-cp313-cp313-win_amd64.whl (506 kB)

   ----------------------------------------  0/21 [zstand

  You can safely remove it manually.

[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Nạp mã nguồn Agent (không gọi API)

Đặt `agent_integration/nhanes_prediction_graph.py` vào thư mục gốc `03-ML-Research`. Đảm bảo chọn đúng Jupyter kernel. Nếu bạn đang chạy notebook trong thư mục `notebooks/`, đoạn code bên dưới sẽ tìm thư mục dự án tự động.

In [3]:
from pathlib import Path
import sys
from pprint import pprint
from datetime import datetime, timezone
import json
import requests

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "agent_integration" / "nhanes_prediction_graph.py").is_file()), None)
assert ROOT is not None, "Không tìm thấy thư mục agent_integration trong 03-ML-Research"
sys.path.insert(0, str(ROOT))
from agent_integration.nhanes_prediction_graph import (
    build_prediction_graph, EXPECTED_SHA256, THRESHOLD, EXPECTED_TASK,
    PREDICT_URL, RESEARCH_WARNING
)
print("Project:", ROOT)
print("SHA256 expected:", EXPECTED_SHA256)
print("Threshold:", THRESHOLD)

Project: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research
SHA256 expected: 31e9c94b0f14369be7920b9315b026d0c4afedb9f7b2885d120d631d930582b1
Threshold: 0.225


## 3. Unit test mô phỏng (không gọi model/API)

Hàm giả trả response **tự tạo để kiểm tra logic định tuyến**. Con số mock không phải dự đoán y khoa. Không cập nhật mô hình hoặc kết quả đã đánh giá.

In [4]:
def fake_predict(features):
    # Mock chỉ kiểm đường đi; các score sau KHÔNG đến từ mô hình thực.
    score = 0.30 if features["RIDAGEYR"] < 50 else 0.40
    return 200, {
        "status": "research_only", "task": EXPECTED_TASK,
        "model_name": "LogisticRegression", "model_sha256": EXPECTED_SHA256,
        "threshold": THRESHOLD, "model_score": score,
        "predicted_label": int(score >= THRESHOLD),
        "warning": RESEARCH_WARNING,
    }

fake_graph = build_prediction_graph(fake_predict)
mock_inputs = [
    {"RIDAGEYR": 45, "RIAGENDR": 1, "BMXBMI": 28.5},
    {"RIDAGEYR": 60, "RIAGENDR": 2, "BMXBMI": 34.8},
]
mock_passed = 0
for i, item in enumerate(mock_inputs, 1):
    s = fake_graph.invoke({"request_id": f"mock-{i}", "task_type": EXPECTED_TASK, "features": item})
    assert s["final_response"]["status"] == "ok"
    assert s["final_response"]["model_sha256"] == EXPECTED_SHA256
    mock_passed += 1
print("MOCK VALID:", mock_passed, "/ 2; không gọi API thật")

MOCK VALID: 2 / 2; không gọi API thật


## 4. Đầu vào sai phải bị từ chối **trước khi gọi API**

Đảm bảo đồ thị kiểm tra sai kiểu, thiếu trường, ngoài miền, trường lạ.

In [5]:
bad_inputs = [
    {"RIDAGEYR": 19, "RIAGENDR": 1, "BMXBMI": 28.5},
    {"RIDAGEYR": 45, "RIAGENDR": 3, "BMXBMI": 28.5},
    {"RIDAGEYR": 45, "RIAGENDR": 1, "BMXBMI": 80.0},
    {"RIDAGEYR": 45, "RIAGENDR": 1},
    {"RIDAGEYR": 45, "RIAGENDR": 1, "BMXBMI": 28.5, "Glucose": 150},
    {"RIDAGEYR": "45", "RIAGENDR": 1, "BMXBMI": 28.5},
]
calls = {"n":0}
def fail_if_called(features):
    calls["n"] += 1
    raise AssertionError("Không được gọi API nếu input sai")
invalid_graph = build_prediction_graph(fail_if_called)
invalid_passed = 0
for i, data in enumerate(bad_inputs, 1):
    s = invalid_graph.invoke({"request_id": f"bad-{i}", "task_type": EXPECTED_TASK, "features": data})
    assert s["final_response"]["status"] == "needs_input"
    assert "model_score" not in s["final_response"]
    invalid_passed += 1
assert calls["n"] == 0
print("INVALID REJECTED BEFORE API:", invalid_passed, "/ 6; api_calls:", calls["n"])

INVALID REJECTED BEFORE API: 6 / 6; api_calls: 0


## 5. Kiểm thử lỗi upstream (không gọi API thật)

Simulate network outage và giả lập API trả sai SHA256. Agent phải từ chối, không tự tạo kết quả.

In [6]:
def fake_network_down(features):
    raise requests.exceptions.ConnectionError("simulated offline")
offline = build_prediction_graph(fake_network_down).invoke({
    "request_id":"offline-mock", "task_type": EXPECTED_TASK,"features":mock_inputs[0]
})
assert offline["final_response"]["error_code"] == "PREDICTION_SERVICE_UNAVAILABLE"

def fake_wrong_sha(features):
    status, d = fake_predict(features)
    return status, dict(d, model_sha256="wrong-hash")
wrong_sha = build_prediction_graph(fake_wrong_sha).invoke({
    "request_id":"sha-mismatch-mock","task_type":EXPECTED_TASK,"features":mock_inputs[0]
})
assert wrong_sha["final_response"]["error_code"] == "UPSTREAM_CONTRACT_MISMATCH"
print("OFFLINE & WRONG_SHA: 2 / 2 handled")

OFFLINE & WRONG_SHA: 2 / 2 handled


## 6. Kiểm thử end-to-end thật: cần bạn khởi động API 8B trước

**Hãy kiểm tra** `http://127.0.0.1:8000/health/ready` trong trình duyệt hoặc terminal. Nếu chưa chạy server, **đừng chạy ô tiếp theo**. Ô này chỉ gọi API 8B bằng hai input giả lập; model được nạp **bên server do bạn điều khiển** và không có `.fit()`.

In [7]:
real_graph = build_prediction_graph()
ready = requests.get("http://127.0.0.1:8000/health/ready", timeout=(2,5))
assert ready.status_code == 200, f"API chưa sẵn sàng: {ready.status_code}"
assert ready.json().get("model_sha256") == EXPECTED_SHA256, "API đang dùng model khác"
real_scores = []
for i, features in enumerate(mock_inputs, 1):
    state = real_graph.invoke({"request_id": f"synthetic-live-{i}",
                               "task_type": EXPECTED_TASK, "features": features})
    result = state["final_response"]
    assert result["status"] == "ok", result
    assert result["model_sha256"] == EXPECTED_SHA256
    assert result["threshold"] == THRESHOLD
    assert result["predicted_label"] == int(result["model_score"] >= THRESHOLD)
    real_scores.append(round(result["model_score"],5))
print("LIVE API THROUGH LANGGRAPH: 2 / 2 | Scores synthetic:", real_scores)

LIVE API THROUGH LANGGRAPH: 2 / 2 | Scores synthetic: [0.22552, 0.39459]


## 7. Lưu báo cáo kỹ thuật – không lưu payload cá thể

Chỉ tạo JSON tổng hợp kiểm thử. **Không** ghi đầu vào giả lập, dữ liệu người dùng hay kết quả xét nghiệm thực tế.

In [8]:
assert mock_passed == 2 and invalid_passed == 6
assert len(real_scores) == 2
report = {
 "experiment": "EXP-DEPLOY-8C-001",
 "stage": "phase3_step8c_langgraph_prediction_agent",
 "created_at_utc": datetime.now(timezone.utc).isoformat(),
 "research_only": True, "model_sha256": EXPECTED_SHA256,
 "threshold": THRESHOLD, "model_task": EXPECTED_TASK,
 "mock_valid_passed": mock_passed,
 "invalid_rejected_before_api": invalid_passed,
 "upstream_error_cases_handled": 2,
 "real_local_api_cases_passed": len(real_scores),
 "real_local_api_scores_synthetic": real_scores,
 "full_five_agents_implemented": False,
 "fit_executed": False, "test_reopened": False,
 "model_modified": False, "clinical_validation": False,
 "api_publicly_exposed": False
}
out = ROOT / "data" / "outputs" / "model_deployment" / "langgraph_8c"
out.mkdir(parents=True, exist_ok=True)
path = out / "08c_prediction_agent_smoke_results.json"
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("===== BƯỚC 8C — LANGGRAPH PREDICTION AGENT (NGƯỜI DÙNG TỰ CHẠY) =====")
print("Model SHA256:", EXPECTED_SHA256)
print("Locked threshold:", THRESHOLD)
print("Mock valid:", mock_passed, "/ 2")
print("Invalid rejected before HTTP:", invalid_passed, "/ 6")
print("Upstream errors handled: 2 / 2")
print("Live synthetic via FastAPI:", len(real_scores), "/ 2")
print("Saved report:", path)
print("FIT: NO | TEST OPENED: NO | MODEL MODIFIED: NO | LLM USED: NO")
print("==== KẾT THÚC — GỬI OUTPUT CHO TÔI ====")

===== BƯỚC 8C — LANGGRAPH PREDICTION AGENT (NGƯỜI DÙNG TỰ CHẠY) =====
Model SHA256: 31e9c94b0f14369be7920b9315b026d0c4afedb9f7b2885d120d631d930582b1
Locked threshold: 0.225
Mock valid: 2 / 2
Invalid rejected before HTTP: 6 / 6
Upstream errors handled: 2 / 2
Live synthetic via FastAPI: 2 / 2
Saved report: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\model_deployment\langgraph_8c\08c_prediction_agent_smoke_results.json
FIT: NO | TEST OPENED: NO | MODEL MODIFIED: NO | LLM USED: NO
==== KẾT THÚC — GỬI OUTPUT CHO TÔI ====
